# 베네픽 RAG 평가 - Ragas 4개 지표 채점 (Colab용)

**측정 지표:**
- **Faithfulness** : context → answer (환각 방지)
- **Answer Relevancy** : answer → question (답변 관련성)
- **Context Precision** : ground_truth + context 순위 (검색 정밀도)
- **Context Recall** : ground_truth → context (필요 문서 회수율)

**사전 준비**: `interim_results_YYYYMMDD_HHMM.json` 파일을 Colab에 업로드하세요.

In [ ]:
# ── 1. 패키지 설치 ──────────────────────────────────
!pip install -q ragas==0.4.3 langchain-ollama langchain-huggingface datasets openpyxl nest_asyncio


In [ ]:
# ── 2. Ollama 설치 및 실행 ─────────────────────
import subprocess, time, urllib.request

# GPU 감지 도구 먼저 설치 (없으면 Ollama가 CPU 모드로 설치됨)
!apt-get install -qq -y zstd lshw pciutils

# Ollama 설치 (GPU 감지 포함)
!curl -fsSL https://ollama.com/install.sh | sh

# GPU 인식 확인
!nvidia-smi | head -5

# 백그라운드로 ollama serve 실행
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 서버 준비 될 때까지 헬스체크 (고정 sleep보다 안정적)
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=1)
        print(f'Ollama 서버 시작 완료 ({i+1}초)')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama 서버가 30초 내 시작되지 않았습니다')


In [ ]:
# ── 3. 모델 다운로드 ───────────────────────────────
# thinking 비활성화는 Cell 6의 CleanChatOllama에서 /no_think 토큰으로 처리
!ollama pull qwen3.5:2b
MODEL_NAME = "qwen3.5:2b"
print(f"사용 모델: {MODEL_NAME}")


In [ ]:
# ── 4. Ground Truth (Reference) 정의 ─────────────────────────
# Context Precision / Context Recall 측정에 필요
# 각 질문에 대한 '이상적인 정답' — 정책명 + 핵심 조건 + 신청 방법 1~2문장
#
# ※ 수정 가이드: 아래 딕셔너리에서 질문 텍스트를 key로 찾아 답변을 수정하세요.
# ※ 질문 텍스트는 interim_results JSON의 question 값과 정확히 일치해야 합니다.

GROUND_TRUTH = {
    # ── 청년 ──────────────────────────────────────────────────
    "서울 청년 월세 지원 받을 수 있나요?":
        "서울시 청년 월세 지원은 서울 거주 만 19~39세 무주택 청년 1인 가구에게 월 최대 20만원을 최대 12개월간 지원한다. "
        "국토교통부 청년 월세 한시 특별지원은 만 19~34세 독립거주 청년에게 월 최대 20만원을 24개월 지원한다. "
        "주민센터 또는 복지로(bokjiro.go.kr)에서 신청 가능하다.",

    "취업 준비생이 받을 수 있는 훈련비 지원이 뭐가 있나요?":
        "국민내일배움카드를 발급받으면 직업훈련 비용을 연간 200~500만원 한도 내에서 지원받을 수 있다. "
        "취업 준비생, 재직자, 자영업자 모두 신청 가능하며 고용24(work.go.kr)에서 신청한다. "
        "일부 훈련 과정은 자부담 없이 전액 지원된다.",

    "청년 창업 지원금 신청 방법 알려주세요":
        "중소벤처기업부 예비창업패키지는 창업 3년 미만 예비창업자에게 최대 1억원 사업화 자금을 지원하며 k-startup.go.kr에서 신청한다. "
        "청년창업사관학교는 만 39세 이하 예비창업자를 대상으로 사업화 자금과 교육을 지원한다. "
        "창업진흥원 홈페이지에서 모집 공고를 확인 후 신청한다.",

    "대학생이 받을 수 있는 장학금 종류":
        "한국장학재단 국가장학금은 가구 소득 분위에 따라 연간 최대 520만원을 지원한다. "
        "근로장학금은 교내외 근로를 통해 학기당 최대 300만원을 지원한다. "
        "한국장학재단 홈페이지(kosaf.go.kr)에서 학기 초에 신청해야 한다.",

    "청년 주거 지원 정책 알려줘":
        "청년 주거 지원 정책으로는 청년 월세 한시 특별지원(월 20만원), 청년 전용 보증금 대출, "
        "LH 청년 매입임대주택, 행복주택 등이 있다. "
        "지역별 지자체 사업도 있으며 복지로 또는 마이홈포털에서 확인 가능하다.",

    "만 34세 미만 청년 고용 지원 프로그램":
        "청년내일채움공제는 중소기업에 취업한 청년이 2~3년 근무 시 1600~3000만원의 목돈을 마련할 수 있는 제도다. "
        "청년 일자리 도약 장려금은 취업 취약 청년을 채용한 기업에 최대 720만원을 지원한다. "
        "고용24에서 신청하며 만 15~34세 미취업 청년이 대상이다.",

    "청년 내일채움공제 조건":
        "청년내일채움공제는 만 15~34세 청년이 중소·중견기업에 정규직으로 취업한 경우 신청 가능하다. "
        "청년이 2년간 300만원 납부 시 기업과 정부가 추가 적립해 만기 시 1200만원을 받는다. "
        "고용24(work.go.kr)에서 취업일로부터 6개월 이내에 신청해야 한다.",

    # ── 노인 ──────────────────────────────────────────────────
    "65세 이상 혼자 사는 노인 지원 정책":
        "노인 맞춤 돌봄 서비스는 65세 이상 기초생활수급자·차상위계층 독거노인에게 안전 확인, 생활 지원 서비스를 제공한다. "
        "기초연금은 만 65세 이상 소득 하위 70%에게 월 최대 33만원을 지급한다. "
        "주민센터 또는 복지로에서 신청한다.",

    "노인 돌봄 서비스 신청 방법":
        "노인 맞춤 돌봄 서비스는 주민센터 또는 노인복지관을 통해 신청한다. "
        "65세 이상 독거노인·취약노인이 대상이며 안전 확인, 생활 교육, 사회 참여, 일상 생활 지원 등을 제공한다. "
        "국민건강보험공단(nhis.or.kr)에서도 신청 가능하다.",

    "기초연금 받으려면 어떻게 해야 하나요?":
        "기초연금은 만 65세 이상 대한민국 국적자 중 가구 소득인정액이 선정기준액 이하인 경우 지급된다. "
        "2024년 기준 단독 가구 213만원, 부부 가구 341만원 이하여야 하며 월 최대 33만 4810원을 받는다. "
        "주민센터, 국민연금공단, 또는 복지로에서 신청할 수 있다.",

    "노인 의료비 지원 제도":
        "의료급여 수급 노인은 본인부담금 없이 의료 서비스를 이용할 수 있다. "
        "65세 이상 건강보험 가입자는 틀니 및 임플란트 건강보험 적용으로 본인부담 30%만 부담한다. "
        "노인 실명 예방 사업으로 백내장 수술 등을 지원받을 수 있으며 보건소 및 복지로에서 신청한다.",

    # ── 장애인 ────────────────────────────────────────────────
    "장애인 취업 지원 프로그램 알려줘":
        "한국장애인고용공단은 장애인 직업훈련, 취업 알선, 보조공학기기 지원 등을 제공한다. "
        "장애인 고용장려금은 장애인을 고용한 사업주에게 지급된다. "
        "발달장애인 일자리 사업, 중증장애인 지역맞춤형 취업지원 등도 있으며 eis.or.kr에서 신청한다.",

    "장애인 의료비 지원 혜택":
        "등록 장애인은 건강보험 본인부담금 경감(외래 15%, 입원 10%), 장애인 의료비 지원, 보조기기 건강보험 적용 혜택을 받는다. "
        "의료급여 수급 장애인은 본인부담 없이 의료를 이용할 수 있다. "
        "장애인 건강 주치의 제도를 통해 만성질환 및 장애 관련 건강 관리 서비스를 받을 수 있다.",

    "장애인 활동 지원 서비스 신청 방법":
        "장애인 활동지원급여는 만 6~65세 미만 등록 장애인이 신청 가능하다. "
        "주민센터 또는 국민건강보험공단에 신청 후 인정조사를 받아 활동지원등급을 받으면 활동보조 서비스를 이용할 수 있다. "
        "복지로(bokjiro.go.kr)에서도 온라인 신청이 가능하다.",

    # ── 저소득 ────────────────────────────────────────────────
    "기초생활수급자 혜택 뭐가 있어요?":
        "기초생활수급자는 생계급여(중위소득 30% 이하), 의료급여(의료비 전액 지원), "
        "주거급여(임차료 지원), 교육급여(교육활동지원비) 등을 받는다. "
        "주민센터 또는 복지로에서 통합 신청할 수 있다.",

    "긴급복지지원 신청 조건":
        "긴급복지지원은 갑작스러운 위기 상황(주 소득자 사망·실직·부상, 가정폭력, 화재 등)으로 "
        "생계 유지가 어려운 저소득 가구에 생계비·의료비·주거비 등을 일시적으로 지원한다. "
        "중위소득 75% 이하가 기준이며 주민센터 또는 보건복지상담센터(129)에서 신청한다.",

    "차상위계층이 받을 수 있는 지원":
        "차상위계층(중위소득 50% 이하 비수급자)은 의료비 지원(본인부담 경감), 교육비 지원, "
        "통신요금 감면(최대 월 1만1000원), 에너지 바우처, 문화누리카드(연 11만원) 등을 받을 수 있다. "
        "주민센터 또는 복지로에서 신청하며 차상위 확인서 발급이 필요하다.",

    "생계급여 받으려면 소득 기준이 어떻게 되나요?":
        "생계급여는 가구 소득인정액이 기준 중위소득 30% 이하인 경우 지급된다. "
        "2024년 기준 1인 가구 71만3102원, 4인 가구 183만3572원 이하여야 한다. "
        "생계급여액은 선정기준액에서 소득인정액을 뺀 금액이며 주민센터 또는 복지로에서 신청한다.",

    # ── 출산·육아 ─────────────────────────────────────────────
    "출산 지원금 얼마나 받을 수 있어요?":
        "출산 시 첫만남이용권으로 첫째 200만원, 둘째 이상 300만원 바우처를 받는다. "
        "부모급여는 0세 월 100만원, 1세 월 50만원을 지급한다. "
        "행복출산 원스톱서비스 또는 복지로에서 신청하며 지자체 추가 지원이 있을 수 있다.",

    "어린이집 보육료 지원 신청":
        "만 0~5세 아동은 어린이집 보육료를 전액 정부 지원받는다. "
        "아이행복카드(보육료 바우처)를 통해 사용하며 복지로 또는 주민센터에서 신청한다. "
        "가정양육수당은 어린이집을 이용하지 않는 가정에 월 10~20만원을 지원한다.",

    "한부모 가정 아동 양육비 지원":
        "한부모 가정 아동 양육비는 중위소득 60% 이하 한부모 가정의 만 18세 미만 아동에게 월 21만원을 지원한다. "
        "청소년 한부모는 월 35만원을 지원하며 학용품비·생활보조금도 별도 지원한다. "
        "주민센터 또는 복지로에서 신청한다.",

    "육아휴직 급여 얼마나 받나요?":
        "육아휴직 급여는 통상임금의 80%(상한 월 150만원, 하한 70만원)를 지급한다. "
        "부모 모두 육아휴직 시 첫 3개월은 통상임금 100%(상한 250만원)를 지급하는 3+3 부모 육아휴직제가 적용된다. "
        "고용보험 피보험자로 180일 이상 가입된 근로자가 대상이며 고용24에서 신청한다.",

    # ── 다문화 ────────────────────────────────────────────────
    "다문화 가정 한국어 교육 지원":
        "다문화가족지원센터에서는 결혼이민자를 대상으로 한국어 교육을 무료 제공한다. "
        "사회통합프로그램(KIIP)은 법무부가 운영하며 한국어와 한국 문화를 단계별로 교육한다. "
        "이민자 사회통합 정보망(socinet.go.kr)에서 신청하며 수료 시 귀화·영주 신청 시 혜택이 있다.",

    "결혼이민자 복지 서비스":
        "결혼이민자는 다문화가족지원센터에서 한국어 교육, 가족 상담, 취업 연계, 자녀 교육 지원 등 다양한 서비스를 받을 수 있다. "
        "다문화가족 자녀 언어 발달 지원, 다문화 가족 방문 교육 서비스도 제공된다. "
        "다누리콜센터(1577-1366)에서 통역 서비스와 함께 안내를 받을 수 있다.",

    # ── 고용·실업 ─────────────────────────────────────────────
    "실직한 30대 신청할 수 있는 복지 정책":
        "실직 시 고용보험 가입자는 구직급여(실업급여)를 신청할 수 있으며 이직 전 평균임금의 60%를 최대 240일간 지급한다. "
        "국민취업지원제도는 취업 취약계층에게 취업 지원 서비스와 구직촉진수당(월 50만원, 6개월)을 제공한다. "
        "고용24(work.go.kr) 또는 고용센터에서 신청한다.",

    "국민취업지원제도 신청 방법":
        "국민취업지원제도는 고용24(work.go.kr) 또는 고용센터에서 신청한다. "
        "1유형은 저소득층 구직자에게 구직촉진수당(월 50만원, 6개월)을 지원하며 2유형은 취업 지원 서비스만 제공한다. "
        "신청 후 취업활동계획 수립 및 구직 활동 이행 요건을 충족해야 수당을 받는다.",

    "소상공인 자금 지원 받을 수 있는 조건":
        "소상공인 정책자금은 상시 근로자 5인 미만(제조·건설업 10인 미만) 소상공인이 신청 가능하다. "
        "직접대출(연 2~3%대 저금리)과 대리대출 방식으로 최대 7000만원~1억원을 지원한다. "
        "소상공인시장진흥공단(sbiz.or.kr) 홈페이지 또는 지역 센터에서 신청한다.",

    # ── 주거 ──────────────────────────────────────────────────
    "전세 사기 피해자 지원 정책":
        "전세 사기 피해자는 전세사기피해자지원특별법에 따라 경매 유예, 피해주택 우선 매수권을 받을 수 있다. "
        "LH에서 피해자를 위한 공공 임대 주택을 우선 공급하며 긴급복지지원도 신청 가능하다. "
        "정부24 또는 주민센터에서 피해자 인정 신청을 할 수 있다.",

    "주거급여 신청 자격":
        "주거급여는 소득인정액이 기준 중위소득 47% 이하인 가구가 신청 가능하다. "
        "2024년 기준 1인 가구 97만원, 4인 가구 255만원 이하여야 한다. "
        "임차 가구는 임차료를, 자가 가구는 주택 수선비를 지원받으며 주민센터 또는 복지로에서 신청한다.",

    # ── 의료 ──────────────────────────────────────────────────
    "암 환자 의료비 지원 방법":
        "암 환자는 건강보험 산정특례에 등록하면 본인부담률이 5%로 낮아진다. "
        "의료급여 수급자는 본인부담 없이 암 치료가 가능하다. "
        "저소득층 암 환자는 국립암센터 지원사업을 통해 입원비·외래비·약제비를 최대 200만원까지 지원받을 수 있으며 주민센터 또는 국립암센터에서 신청한다.",
}

print(f'Ground Truth 정의 완료: {len(GROUND_TRUTH)}개 질문')

In [ ]:
# ── 5. interim JSON 업로드 확인 ──────────────────────────────
import json, glob

# 업로드된 파일 자동 탐색
files = glob.glob('interim_results_*.json')
if not files:
    # Colab 파일 업로드 UI 사용
    from google.colab import files
    uploaded = files.upload()
    interim_path = list(uploaded.keys())[0]
else:
    interim_path = sorted(files)[-1]  # 가장 최신 파일

with open(interim_path, 'r', encoding='utf-8') as f:
    interim = json.load(f)

dataset_rows = interim['rows']
categories   = interim['categories']

# Ground Truth 병합
# 질문과 일치하는 reference를 각 row에 추가
matched, unmatched = 0, []
for row in dataset_rows:
    q = row['question']
    ref = GROUND_TRUTH.get(q, '')
    row['reference'] = ref
    if ref:
        matched += 1
    else:
        unmatched.append(q)

print(f'로드 완료: {len(dataset_rows)}개 Q&A 쌍')
print(f'Ground Truth 매칭: {matched}/{len(dataset_rows)}개')
if unmatched:
    print(f'\n⚠️  미매칭 질문 ({len(unmatched)}개) — Context Precision/Recall 측정 제외:')
    for q in unmatched:
        print(f'  - {q}')
print(f'\n샘플: {dataset_rows[0]["question"][:50]}')
print(f'참조 샘플: {dataset_rows[0]["reference"][:60]}...')

In [ ]:
# ── 6. Ragas 평가 실행 (4개 지표) ────────────────────────────────
# Colab/Jupyter 이벤트 루프 충돌 해결 (이게 없으면 0/60에서 영원히 대기함)
import nest_asyncio
nest_asyncio.apply()

import warnings, numpy as np, json as _json
warnings.filterwarnings('ignore')

from datasets import Dataset
from ragas import evaluate
from ragas.metrics._faithfulness import Faithfulness
from ragas.metrics._answer_relevance import AnswerRelevancy
from ragas.metrics._context_recall import ContextRecall
from ragas.metrics._context_precision import ContextPrecision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult


# ── JSON 스키마 echo 수정 ─────────────────────────────────────
def _strip_think(content: str) -> str:
    """<think>...</think> 블록 제거 — qwen3 계열 모델 thinking 토큰 처리"""
    import re as _re
    return _re.sub(r'<think>.*?</think>', '', content, flags=_re.DOTALL).strip()


def _extract_last_json(content: str) -> str:
    # 1단계: thinking 블록 먼저 제거 (JSON 추출 전 필수)
    content = _strip_think(content)
    depth, start, objects = 0, -1, []
    for i, c in enumerate(content):
        if c == '{':
            if depth == 0:
                start = i
            depth += 1
        elif c == '}':
            depth -= 1
            if depth == 0 and start != -1:
                candidate = content[start:i+1]
                try:
                    parsed = _json.loads(candidate)
                    if not ('properties' in parsed and 'type' in parsed):
                        objects.append(candidate)
                except Exception:
                    pass
                start = -1
    return objects[-1] if objects else content

def _clean_result(result: ChatResult) -> ChatResult:
    cleaned = []
    for gen in result.generations:
        cleaned_content = _extract_last_json(gen.message.content)
        cleaned.append(ChatGeneration(message=AIMessage(content=cleaned_content)))
    return ChatResult(generations=cleaned)

class CleanChatOllama(ChatOllama):
    """Ragas 평가용 — JSON 스키마 echo 제거 + Qwen3 thinking 비활성화(/no_think)"""

    @staticmethod
    def _inject_no_think(messages):
        """첫 번째 SystemMessage 앞에 /no_think 삽입 → Qwen3 thinking 모드 OFF"""
        from langchain_core.messages import SystemMessage as SM
        msgs = list(messages)
        for i, msg in enumerate(msgs):
            if isinstance(msg, SM):
                msgs[i] = SM(content="/no_think
" + msg.content)
                return msgs
        msgs.insert(0, SM(content="/no_think"))
        return msgs

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        return _clean_result(super()._generate(self._inject_no_think(messages), stop, run_manager, **kwargs))

    async def _agenerate(self, messages, stop=None, run_manager=None, **kwargs):
        return _clean_result(await super()._agenerate(self._inject_no_think(messages), stop, run_manager, **kwargs))


# ── LLM / 임베딩 초기화 ──────────────────────────────────────
ragas_llm = LangchainLLMWrapper(
    CleanChatOllama(model=MODEL_NAME, temperature=0, num_ctx=4096, num_predict=512)
)
ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        model_kwargs={'device': 'cpu'},
    )
)

# ── 지표별 데이터셋 분리 ─────────────────────────────────────
# Faithfulness / AnswerRelevancy : reference 불필요 (전체 30개)
# ContextPrecision / ContextRecall : reference 필수 (매칭된 것만)

rows_all      = dataset_rows                           # 전체
rows_with_ref = [r for r in dataset_rows if r.get('reference')]  # reference 있는 것만

dataset_all      = Dataset.from_list(rows_all)
dataset_with_ref = Dataset.from_list(rows_with_ref)

print(f'전체 데이터셋     : {len(rows_all)}개 (Faithfulness, Answer Relevancy용)')
print(f'Reference 포함    : {len(rows_with_ref)}개 (Context Precision, Context Recall용)')

run_cfg = RunConfig(timeout=600, max_retries=2, max_workers=1)

# ── Faithfulness + Answer Relevancy (전체) ──────────────────
print('\n[1/2] Faithfulness + Answer Relevancy 평가 중... (30~50분 소요 예상)')
result_fa = evaluate(
    dataset=dataset_all,
    metrics=[Faithfulness(), AnswerRelevancy()],
    llm=ragas_llm,
    embeddings=ragas_emb,
    raise_exceptions=False,
    run_config=run_cfg,
)
print('완료!')

# ── Context Precision + Context Recall (reference 있는 것만) ──
print('\n[2/2] Context Precision + Context Recall 평가 중... (20~40분 소요 예상)')
result_cr = evaluate(
    dataset=dataset_with_ref,
    metrics=[ContextPrecision(), ContextRecall()],
    llm=ragas_llm,
    embeddings=ragas_emb,
    raise_exceptions=False,
    run_config=run_cfg,
)
print('평가 완료!')

In [ ]:
# ── 7. 결과 출력 및 저장 ─────────────────────────────────────
import pandas as pd
from datetime import datetime

def safe_score(val):
    if isinstance(val, list):
        vals = [float(v) for v in val if v is not None and not (isinstance(v, float) and np.isnan(v))]
        return float(np.mean(vals)) if vals else float('nan')
    if val is None:
        return float('nan')
    try:
        v = float(val)
        return v
    except:
        return float('nan')

# ── 점수 추출 ─────────────────────────────────────────────────
df_fa = result_fa.to_pandas()
df_cr = result_cr.to_pandas()

faith_scores  = [safe_score(v) for v in df_fa['faithfulness']]
relev_scores  = [safe_score(v) for v in df_fa['answer_relevancy']]

# Context Precision/Recall: reference 있는 행만 → 전체 크기 맞춰 NaN 채우기
ref_indices = [i for i, r in enumerate(dataset_rows) if r.get('reference')]
cp_scores_partial = [safe_score(v) for v in df_cr['context_precision']]
cr_scores_partial = [safe_score(v) for v in df_cr['context_recall']]

cp_scores = [float('nan')] * len(dataset_rows)
cr_scores = [float('nan')] * len(dataset_rows)
for i, (cp, cr) in zip(ref_indices, zip(cp_scores_partial, cr_scores_partial)):
    cp_scores[i] = cp
    cr_scores[i] = cr

# ── 지표별 평균 (NaN 제외) ────────────────────────────────────
def nanmean(lst):
    vals = [v for v in lst if not np.isnan(v)]
    return round(float(np.mean(vals)), 4) if vals else float('nan')

faith_avg = nanmean(faith_scores)
relev_avg = nanmean(relev_scores)
cp_avg    = nanmean(cp_scores)
cr_avg    = nanmean(cr_scores)

# ── 결과 출력 ─────────────────────────────────────────────────
print('=' * 55)
print('평가 결과 요약')
print('=' * 55)
print(f'  Faithfulness       (환각 방지):   {faith_avg:.4f}  ({sum(1 for v in faith_scores if not np.isnan(v))}/{len(faith_scores)}개)')
print(f'  Answer Relevancy   (답변 관련성): {relev_avg:.4f}  ({sum(1 for v in relev_scores if not np.isnan(v))}/{len(relev_scores)}개)')
print(f'  Context Precision  (검색 정밀도): {cp_avg:.4f}  ({sum(1 for v in cp_scores if not np.isnan(v))}/{len(cp_scores)}개)')
print(f'  Context Recall     (문서 회수율): {cr_avg:.4f}  ({sum(1 for v in cr_scores if not np.isnan(v))}/{len(cr_scores)}개)')
print('-' * 55)
all_avgs = [v for v in [faith_avg, relev_avg, cp_avg, cr_avg] if not np.isnan(v)]
total_avg = round(float(np.mean(all_avgs)), 4) if all_avgs else float('nan')
print(f'  4개 지표 평균:                   {total_avg:.4f}')

# ── 카테고리별 평균 ───────────────────────────────────────────
df_detail = pd.DataFrame({
    '카테고리':          categories,
    '질문':             [r['question'] for r in dataset_rows],
    'Faithfulness':     [round(s, 4) if not np.isnan(s) else None for s in faith_scores],
    'Answer Relevancy': [round(s, 4) if not np.isnan(s) else None for s in relev_scores],
    'Context Precision':[round(s, 4) if not np.isnan(s) else None for s in cp_scores],
    'Context Recall':   [round(s, 4) if not np.isnan(s) else None for s in cr_scores],
    '답변':             [r['answer'] for r in dataset_rows],
    'Reference':        [r.get('reference', '') for r in dataset_rows],
})

print('\n카테고리별 평균')
print('-' * 55)
cat_stats = df_detail.groupby('카테고리')[['Faithfulness','Answer Relevancy','Context Precision','Context Recall']].mean().round(4)
print(cat_stats.to_string())

# ── NaN 발생 질문 (4개 지표 중 하나라도) ─────────────────────
nan_mask = df_detail[['Faithfulness','Answer Relevancy','Context Precision','Context Recall']].isna().any(axis=1)
if nan_mask.any():
    print(f'\n⚠️  점수 미산출 질문 ({nan_mask.sum()}개):')
    for _, row in df_detail[nan_mask].iterrows():
        missing = [col for col in ['Faithfulness','Answer Relevancy','Context Precision','Context Recall'] if pd.isna(row[col])]
        print(f'  [{row["카테고리"]}] {row["질문"][:40]} → {missing}')

# ── Excel 저장 ────────────────────────────────────────────────
timestamp  = datetime.now().strftime('%Y%m%d_%H%M')
excel_path = f'실험결과_Ragas4_{timestamp}.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # 시트 1: 질문별 상세
    df_detail.drop(columns=['Reference']).to_excel(writer, sheet_name='질문별_상세', index=False)

    # 시트 2: 카테고리별 집계
    cat_stats.reset_index().to_excel(writer, sheet_name='카테고리별_집계', index=False)

    # 시트 3: 요약
    summary_df = pd.DataFrame([
        {'지표': 'Faithfulness',       '점수': faith_avg, '측정수': sum(1 for v in faith_scores if not np.isnan(v)),  '설명': '환각 방지 (context→answer)'},
        {'지표': 'Answer Relevancy',   '점수': relev_avg, '측정수': sum(1 for v in relev_scores if not np.isnan(v)), '설명': '답변 관련성 (answer→question)'},
        {'지표': 'Context Precision',  '점수': cp_avg,    '측정수': sum(1 for v in cp_scores if not np.isnan(v)),    '설명': '검색 정밀도 (ground_truth→context 순위)'},
        {'지표': 'Context Recall',     '점수': cr_avg,    '측정수': sum(1 for v in cr_scores if not np.isnan(v)),    '설명': '문서 회수율 (ground_truth→context 포함여부)'},
        {'지표': '4개 지표 평균',        '점수': total_avg, '측정수': len(dataset_rows),                                '설명': '전체 종합 점수'},
    ])
    summary_df.to_excel(writer, sheet_name='요약', index=False)

print(f'\nExcel 저장 완료: {excel_path} (시트 3개: 질문별_상세 / 카테고리별_집계 / 요약)')

# Colab에서 자동 다운로드
from google.colab import files
files.download(excel_path)